# **FCC-hh di-Higgs Analysis Project Part 1**

## This notebook is separated into different sections:
- **Section a)** Plotting basic kinematics of objects involved in the final-states of signal and background processes
- **Section b)** Calculating complex quantities using these basic kinematics and comparing plots of signal and backgorund processes
- **Section c)** Using a cut-based approach for classification on these complex variables

For some of you, this may be your first time touching Python or any coding. 
To run code in cells:
- Click on a cell
- Either press the triangular  "Run this cell and advance" button or use Ctrl+Enter

Make sure that cells labelled "RUN THIS FIRST" are ran first.

To save something, press the save icon or use Ctrl+S

Hope you have fun :)

In [ ]:
# RUN THIS FIRST
# Initialsing packages used in the code
import sys
sys.path.insert(0, 'src')
from sparticles.dataset_edge_embed import EventsDataset
from sparticles.dataset_edge_embed import DEFAULT_EVENT_SUBSETS
from sparticles.transforms import MakeHomogeneous
import os
import errno
import numpy as np
from numpy import pi
from sparticles.plotting import plot_kinematics, plot_complex_variable, plot_all_kinematics, plot_all_complex_variables

The cell below will initialise download all the files we need for plotting and later on, the machine learning.

In [ ]:
# RUN THIS SECOND

sig_no = 20000 # this is the number of signal events
ttb_no = 20000 # this is the number of background events
               # we have 20000 of each for the plotting
vjets_no = 0

dataset = EventsDataset(
    root='./fcc_hh_data',
    event_subsets={'signal': sig_no, 'vjets': vjets_no, 'ttbar': ttb_no},
    url="https://cernbox.cern.ch/s/QoROVFN6wOrHWx0/download",
    delete_raw_archive=True,
    delete_processed=True,
    add_edge_index=True,
    signal_filename='hhbbtata.h5',
    url="https://cernbox.cern.ch/s/QoROVFN6wOrHWx0/download",
    delete_raw_archive=True,
    delete_processed=True,
    background_filename='mgp8_pp_tt012j_5f.h5',
    m_bb=True,
    dR_bb=True,
    m_tt=True,
    dR_tt=True,
    dpT_tt=True,
    m_HH=True,
    dPhi_HH=True,
)

If a "FutureWarning" message shows up just ignore it.

## **Section a)**

The cell below will show some plots of kinematic quantities of the objects involved in each event. 

Have a go at changing the object that's plotted.

**What do you notice about the differences in shape between signal and background for each quantity?**

In [ ]:
# Plots pT, eta, phi (pT, phi only for 'energy'/MET) for each object, signal vs background, as subplots in one figure per object.
# Equivalently: plot_all_kinematics(dataset)
# Objects:
#  'tau1'   - Highest transverse momentum tau in event
#  'tau2'   - The other tau in event
#  'b1'     - Highest transverse momentum bottom quark in event
#  'b2'     - The other bottom quark in event
#  'energy' - Missing Energy

for obj in ['tau1']: # < --- change what's in the square brackets to look at other objects and then re-run this cell
    plot_kinematics(dataset, obj)

Double-click on this cell here to make notes about what you see that's different for each object for section a):

*e.g. Tau1 pT shows the majority of background tau1 objects have low momentum compared to the more uniform shape of the signal which peaks under 100 GeV*

[placeholder text]

Running this cell or using Ctrl+Enter will return this cell to a fixed state.


## **Section b)**

When we try and distinguish signal and background within real analyses, we look at variables that we calculate from the combination of two different object's quantities. We refer to these as higher-order or complex values.

For example, we often use invariant masses of two different objects, which are basically the combination of the masses of two different objects, we use this to see which objects decay from each particle. For example between the b-quarks, $m_{bb}$

We also use $\Delta R$ which is the angular separation of the objects that we measure within the detector, this is calculated from the object's $\eta$ and $\phi$ values.

Change the variable that's plotted again below.

**What do you notice about the differences in shape between signal and background for each higher order variable? Do you notice any peaks that may correspond to particles within the processes?**

In [ ]:
# Plots each derived variable, signal vs background, as a single histogram.
# Equivalently: plot_all_complex_variables(dataset)
# Variables:
#  'm_bb'    - Invariant mass of the two b-jets
#               (reconstructed mass of the H -> bb candidate)
#  'dR_bb'   - Angular separation between the two b-jets
#               (how close together they are in the detector)
#  'm_tt'    - Invariant mass of the tau system
#               (reconstructed mass of the H -> tautau candidate)
#  'dR_tt'   - Angular separation between the taus
#  'dpT_tt'  - Difference in transverse momentum between the taus
#  'm_HH'    - Invariant mass of the full HH system
#               (combined mass of both Higgs candidates)
#  'dPhi_HH' - Azimuthal angle between the two Higgs candidates
#               (how far apart they are in the transverse plane)
for var in ['m_bb']:  # < --- change what's in the square brackets to look at other variables
    plot_complex_variable(dataset, var)

Double-click on this cell here to make notes about what you see that's different for each object for section b):

[placeholder text]

Running this cell or using Ctrl+Enter will return this cell to a fixed state.

## **Section c)**

Now that we can see that signal and background have different shapes in these variables, we can try to use that to our advantage. A cut is simply a requirement that a variable falls within a certain range, the events that don't satisfy it are rejected.

For example, cutting on $m_{bb}$ between 100 and 140 GeV keeps events where the b-jet pair mass is consistent with a Higgs boson decay, while rejecting most background events where $m_{bb}$ takes a more spread-out distribution.

To measure how well our cuts work, we use a quantity called significance:

$$\frac{S}{\sqrt{B}}$$

where $S$ is the number of signal events passing the cuts and $B$ is the number of background events passing. A higher value means our signal stands out more clearly above the background. 

In particle physics:

- $S\sqrt{B} \geq 3$ is called evidence (3σ) ~ 99.73% probability that the signal is real, rather than a statistical fluctuation in the background
- $S\sqrt{B} \geq 5$ is called  discovery (5σ) ~ 99.99994% probability that the signal is real

Note that the S and B values here are weighted by the expected number of events at the FCC-hh so, a single simulated event may represent many real expected events depending on how rare the process is. The background production rate is much larger than the signal, so even after good cuts, a high significance can be challenging to achieve.

**See how large of a significance value you can get by applying these cuts!**

In [ ]:
from sparticles.analysis import summarise, apply_cuts

# Show the range of each variable to help you choose cut values
summarise(dataset)

Define your cuts below and run the cell.

In [ ]:
# Format: 'variable': (minimum, maximum)
# Use None for no limit on that side.
#
# Available variables:
#   'm_bb'    -- invariant mass of the two b-jets (GeV)
#   'dR_bb'   -- angular separation of the two b-jets
#   'm_tt'    -- invariant mass of the two taus (GeV)
#   'dR_tt'   -- angular separation of the two taus
#   'dpT_tt'  -- pT difference between the two taus (GeV)
#   'm_HH'    -- invariant mass of the full HH system (GeV)
#   'dPhi_HH' -- angle between the two Higgs candidates

cuts = {
    'm_bb':    [(None, None)],  # simply replacee these None values with your cuts (lower cut on left, upper cut on right)
    'dR_bb':   [(None, None)],
    'm_tt':    [(None, None)],
    'dR_tt':   [(None, None)],
    'dpT_tt':  [(None, None)],
    'm_HH':    [(None, None)],
    'dPhi_HH': [(None, None), (None, None)], # hint: you might want two cuts for this variable!
}
print(cuts)
apply_cuts(dataset, cuts);